# Tabla de desempleo regional desde los microdatos de la ENE
Calcula ocupados, desocupados y tasa de desocupación por región (división de 15 regiones, Ñuble dentro de Biobío) y para el total país (código 99), con el factor de expansión calibrado `fact_cal`.

**Convención de fechas:** cada dato es un trimestre móvil y se fecha por su **último mes**, igual que la cifra oficial publicada (los microdatos traen el mes central).

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

print("DuckDB", duckdb.__version__)

DuckDB 1.5.5


In [ ]:


pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


RUTA_PARQUET = Path("ene_cruda.parquet")
PARQ = RUTA_PARQUET.as_posix()

con = duckdb.connect()
PARQ = RUTA_PARQUET.as_posix()




In [3]:
serie = con.sql(f"""
with base as (
    select
        cast(ano_trimestre as int)                         as annio,
        cast(mes_central  as int)                          as mes,
        coalesce(
            cast(region_15 as int),
            case when cast(region as int) = 16 then 8      -- Ñuble vuelve a Biobío
                 else cast(region as int) end
        )                                                  as region_15h,
        activ,
        cast(replace(fact_cal, ',', '.') as double)        as fact
    from '{PARQ}'
),
regional as (
    select
        annio,
        mes,
        region_15h                                         as region_15,
        sum(fact)                                          as poblacion,
        sum(case when activ = '1' then fact else 0 end)    as ocupados,
        sum(case when activ = '2' then fact else 0 end)    as desocupados
    from base
    group by all
),
nacional as (
    select
        annio,
        mes,
        99                                                 as region_15,
        sum(poblacion)                                     as poblacion,
        sum(ocupados)                                      as ocupados,
        sum(desocupados)                                   as desocupados
    from regional
    group by all
),
unido as (
    select * from regional
    union all
    select * from nacional
)
select
    annio,
    mes                                                      as mes_central,
    -- Fecha = último mes del trimestre móvil (convención de la cifra oficial publicada).
    -- Ej.: mes central mayo (abr-may-jun) -> 2026-06-01.
    cast(make_date(annio, mes, 1) + interval 1 month as date) as fecha,
    region_15,
    poblacion,                                               -- población total (todas las edades)
    ocupados,
    desocupados,
    desocupados / (ocupados + desocupados)                 as tasa_desempleo
from unido
order by fecha, region_15
""").df()
serie.to_csv("serie_desempleo_regional_actualizado.csv", index=False, encoding="utf-8-sig")
serie

,annio,mes_central,fecha,region_15,poblacion,ocupados,desocupados,tasa_desempleo
0,2010,2,2010-03-01,1,"291,760.000","135,903.591","10,037.281",0.069
1,2010,2,2010-03-01,2,"559,452.001","238,440.263","22,602.339",0.087
2,2010,2,2010-03-01,3,"284,706.000","117,058.364","11,780.992",0.091
3,2010,2,2010-03-01,4,"702,640.000","284,381.615","32,324.298",0.102
4,2010,2,2010-03-01,5,"1,739,119.000","712,319.152","89,748.022",0.112
...,...,...,...,...,...,...,...,...
3179,2026,8,2026-09-01,12,"185,278.000","98,572.395","6,788.231",0.064
3180,2026,8,2026-09-01,13,"8,525,283.000","4,116,348.626","475,260.687",0.104
3181,2026,8,2026-09-01,14,"415,888.000","181,325.431","19,793.703",0.098
3182,2026,8,2026-09-01,15,"265,760.000","119,895.897","10,118.409",0.078


## Control: factor de expansión
Verifica que `fact_cal` exista en todos los años. Si un año tiene filas sin factor, su población y su tasa quedan mal calculadas.

In [4]:
con.sql(f"""
select
    cast(ano_trimestre as int)                                              as annio,
    count(*)                                                                as filas,
    count(*) filter (where fact_cal is null
                        or trim(cast(fact_cal as varchar)) = '')            as sin_fact_cal,
    min(cast(mes_central as int))                                           as primer_mes_central,
    max(cast(mes_central as int))                                           as ultimo_mes_central
from '{PARQ}'
group by all
order by annio
""").df()

,annio,filas,sin_fact_cal,primer_mes_central,ultimo_mes_central
0,2010,1239296,0,2,12
1,2011,1360039,0,1,12
2,2012,1313146,0,1,12
3,2013,1291698,0,1,12
4,2014,1288335,0,1,12
5,2015,1262135,0,1,12
6,2016,1247349,0,1,12
7,2017,1233941,0,1,12
8,2018,1234721,0,1,12
9,2019,1243786,0,1,12


## Control: último dato disponible
La tabla debe llegar al último trimestre publicado por el INE. Si no, faltan archivos de microdatos por descargar.

In [5]:
print("Último trimestre en la tabla (último mes):", serie["fecha"].max().date())

Último trimestre en la tabla (último mes): 2026-09-01
